In [ ]:
from datasets import load_dataset
import tqdm
dataset = load_dataset("5CD-AI/LLaVA-CoT-o1-Instruct",token="")

In [11]:
!mkdir MathVista

In [3]:
dataset["train"]

Dataset({
    features: ['id', 'image', 'question', 'output', 'ground_truth'],
    num_rows: 58468
})

In [13]:
train_list = []
for i, row in enumerate(tqdm.tqdm(dataset["train"])):
    ans = row["ground_truth"]
        
    row['image'].save("MathVista/"+row['id'].replace("/","____"))
    
    train_list.append({
          "id": i,
          "image": "MathVista/"+row['id'].replace("/","____"),
          "conversations": [
                {
                    'from': 'human', 
                    'value': row["question"].replace("""Please answer the question below, explaining your reasoning step by step before providing the final answer.\n\nQuestion:\n\n""","")
                }, 
                {
                    'from': 'gpt', 
                    'value': "<answer>"+str(ans)+"</answer>"
                }
          ]
        })
    if len(train_list) >= 500:
        break

  1%|          | 499/58468 [00:17<33:59, 28.42it/s]  


In [14]:
train_list

[{'id': 0,
  'image': 'MathVista/Geometry3K____train____1834____img_diagram.png',
  'conversations': [{'from': 'human', 'value': 'Find m \\angle H.'},
   {'from': 'gpt', 'value': '<answer>106</answer>'}]},
 {'id': 1,
  'image': 'MathVista/ai2d____abc_images____3924.png',
  'conversations': [{'from': 'human',
    'value': 'In the diagram, is the Abdomen spicala located at B, C, E, or G?\nB\nE\nG\nC\nPlease answer the question based on the options mentioned before.'},
   {'from': 'gpt', 'value': '<answer>E</answer>'}]},
 {'id': 2,
  'image': 'MathVista/GeomVerse____TRAIN____TRAIN_MIX____TRAIN_MIX_2____images____2286.jpeg',
  'conversations': [{'from': 'human',
    'value': 'If the area of the ABCD parallelogram is 102, compute the degree of the DAB angle. Round computations to 2 decimal places.'},
   {'from': 'gpt', 'value': '<answer>41.3</answer>'}]},
 {'id': 3,
  'image': 'MathVista/geoqa_plus____images____9762.png',
  'conversations': [{'from': 'human',
    'value': 'As shown in the f

In [16]:
import json
with open("MathVista/MathVista.jsonl", "w") as outfile:
    for row in train_list:
        json.dump(row, outfile, ensure_ascii=False)
        outfile.write('\n')

In [47]:
import pandas as pd

In [48]:
MathVista_MINI = pd.read_csv('MathVista_MINI.tsv', sep='\t')

In [49]:
len(set(MathVista_MINI["question"]))

46

In [50]:
from datasets import load_dataset
import tqdm
dataset = load_dataset("AI4Math/MathVista")

In [51]:
count = 0
for row in dataset["testmini"]:
    if row["query"] in set(MathVista_MINI["question"]):
        count += 1

In [52]:
def convert_answer_to_abcd(row):
    choices = row["choices"]
    answer = row["answer"]
    
    if answer in choices:
        return chr(65 + choices.index(answer))  # Chuyển vị trí thành A, B, C, D
    else:
        return ""  # Trả về None nếu không tìm thấy đáp án trong choices

In [53]:
PROMPT = "<image>/nAnswer the following question with short reasoning. The final answer is placed in boxed{...} \nQuestion: "

In [54]:
train_list = []
dict_ = {}
for i, row in enumerate(tqdm.tqdm(dataset["testmini"])):
    if row["query"] not in set(MathVista_MINI["question"]):
        if row["choices"] is not None:
            ans = convert_answer_to_abcd(row)
        else:
            ans = row["answer"]
            
        row['decoded_image'].convert("RGB").save("MathVista/"+row['image'].replace("/","____"))
        
        train_list.append({
              "id": i,
              "image": "MathVista/"+row['image'].replace("/","____"),
              "conversations": [
                    {
                        'from': 'human', 
                        'value': PROMPT + row["query"]    #+ translate_en2vi([row["question"]])[0]
                    }, 
                    {
                        'from': 'gpt', 
                        'value': "<answer>"+str(ans)+"</answer>"
                    }
              ]
            })
        if len(train_list) >= 500:
            break

 55%|█████▌    | 554/1000 [00:08<00:07, 62.82it/s]


In [55]:
len(train_list)

500

In [57]:
train_list[:10]

[{'id': 0,
  'image': 'MathVista/images____1.jpg',
  'conversations': [{'from': 'human',
    'value': "<image>/nAnswer the following question with short reasoning. The final answer is placed in boxed{...} \nQuestion: Hint: Please answer the question requiring a floating-point number with one decimal place and provide the final value, e.g., 1.2, 1.3, 1.4, at the end.\nQuestion: When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. However, we can split the displacement up into an infinite number of tiny parts and then approximate the force in each as being constant. Integration sums the work done in all those parts. Here we use the generic result of the integration.\r\n\r\nIn Figure, a cumin canister of mass $m=0.40 \\mathrm{~kg}$ slides across a horizontal frictionless counter with speed $v=0.50 \\mathrm{~m} / \\mathrm{s}$. It then runs into an

In [58]:
import json
with open("MathVista/MathVista.jsonl", "w") as outfile:
    for row in train_list:
        json.dump(row, outfile, ensure_ascii=False)
        outfile.write('\n')